In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / "Case Study Data Files").is_dir():
    root = root.parent

folder = root / "Case Study Data Files"

sw = pd.read_csv(
    folder / "southwest_quarterly_stats_2019Q1_2026Q2.csv"
)
tsa = pd.read_csv(
    folder / "tsa_daily_checkpoint_2019-01-01_2026-09-30.csv"
)
cpi = pd.read_csv(
    folder / "bls_airline_fares_cpi_CUUR0000SETG01_2018-01_2026-09.csv"
)

In [2]:
sw.index = pd.PeriodIndex(sw["quarter"], freq="Q")

tsa["quarter"] = pd.to_datetime(tsa["date"]).dt.to_period("Q")
cpi["quarter"] = pd.to_datetime(
    cpi["observation_date"]
).dt.to_period("Q")

tsa_q = tsa.groupby("quarter")["passengers_screened"].sum()

cpi_q = cpi.groupby("quarter")["CUUR0000SETG01"].agg(
    ["mean", "count"]
)
cpi_q = cpi_q["mean"].where(cpi_q["count"].eq(3))

data = pd.DataFrame({
    "passengers": sw["revenue_passengers_000"] / 1000,
    "fare": sw["avg_passenger_fare_usd"],
    "tsa": tsa_q,
    "cpi": cpi_q,
}).sort_index()

data.tail()

,passengers,fare,tsa,cpi
quarter,,,,
2025Q3,34.581,182.56,237515574.0,252.289333
2025Q4,34.033,199.36,228297338.0,NaN
2026Q1,29.175,225.93,208660296.0,280.679000
2026Q2,34.331,225.61,233763292.0,320.703000
2026Q3,NaN,NaN,230749661.0,312.286667


In [3]:
yoy = data.pct_change(periods=4, fill_method=None)

pred = pd.DataFrame(index=data.index)

pred["passengers"] = data["passengers"].shift(4) * (
    1
    + 0.5 * yoy["passengers"].shift(1)
    + 0.5 * yoy["tsa"]
)

pred["fare"] = data["fare"].shift(4) * (
    1
    + 0.5 * yoy["fare"].shift(1)
    + 0.5 * yoy["cpi"]
)

pred.loc["2026Q3"]

passengers     33.515794
fare          223.320585
Name: 2026Q3, dtype: float64

In [4]:
errors = (
    data[["passengers", "fare"]] / pred - 1
).loc["2023Q1":"2026Q2"]

margin = errors.abs().quantile(0.80)
point = pred.loc["2026Q3"]

forecast = pd.DataFrame({
    "point": point,
    "lower": point * (1 - margin),
    "upper": point * (1 + margin),
})

forecast.round(2)

,point,lower,upper
passengers,33.52,31.62,35.41
fare,223.32,205.77,240.88


In [5]:
pax = forecast.loc["passengers"]
fare = forecast.loc["fare"]

print(
    "Revenue Passengers Count:",
    f"{np.floor(pax['lower'] * 10) / 10:.1f}–"
    f"{np.ceil(pax['upper'] * 10) / 10:.1f} million passengers"
)

print(
    "Average Passenger Fare:",
    f"${np.floor(fare['lower']):.0f}–"
    f"${np.ceil(fare['upper']):.0f}"
)

print(
    "\nApproach:",
    "We combined Southwest's latest reported year-over-year growth "
    "with Q3 year-over-year changes in TSA throughput and the BLS "
    "airline-fare index, using equal weights. Ranges use the 80th "
    "percentile of absolute percentage errors from historical "
    "predictions since 2023."
)

print("\nOutside data sources: No")

Revenue Passengers Count: 31.6–35.5 million passengers
Average Passenger Fare: $205–$241

Approach: We combined Southwest's latest reported year-over-year growth with Q3 year-over-year changes in TSA throughput and the BLS airline-fare index, using equal weights. Ranges use the 80th percentile of absolute percentage errors from historical predictions since 2023.

Outside data sources: No
